# Tema 1 · Las imágenes como datos

Clase de 150 minutos. Alterna el PDF con las actividades 1–6 de este notebook. Ejecuta por orden; solo necesitas NumPy, Matplotlib y Pillow. No se descargan datos ni se requiere GPU.

**Rutina:** predice el resultado, ejecuta, modifica y explica. Guarda tu copia antes de empezar. Las imágenes son sintéticas: sus conclusiones no validan un sistema de inspección real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def mostrar(imagen, titulo="", gris=False):
    plt.figure(figsize=(6, 4))
    plt.imshow(imagen, cmap="gray" if gris else None,
               vmin=0 if gris else None, vmax=255 if gris else None)
    plt.title(titulo)
    plt.axis("off")
    plt.show()

imagen = np.full((240, 320, 3), 235, dtype=np.uint8)
imagen[40:200, 60:260] = [55, 125, 190]
imagen[100:112, 170:190] = [220, 55, 40]
referencia = imagen.copy()

## Actividad 1 · Leer una imagen (10 min)

**Antes de ejecutar:** ¿qué significan los tres números de su forma? ¿Una marca roja basta para afirmar que hay un defecto real?

In [ ]:
mostrar(imagen, "Pieza sintética")
print("Forma:", imagen.shape)
print("Tipo:", imagen.dtype)
print("Rango observado:", imagen.min(), imagen.max())

**Comprueba:** identifica fondo, pieza y marca. Anota alto, ancho y número de canales.

**Tu explicación:** …

## Actividad 2 · Coordenadas y píxeles (10 min)

NumPy usa `[fila, columna, canal]`; el origen está arriba a la izquierda. Predice el color en cada posición.

In [ ]:
for fila, columna in [(0, 0), (60, 80), (105, 180)]:
    print((fila, columna), imagen[fila, columna].tolist())
experimento = imagen.copy()
experimento[60:80, 80:100] = [255, 255, 0]
mostrar(experimento, "Región modificada")

**Modifica:** desplaza el cuadrado amarillo a la esquina inferior derecha de la pieza. Indica los intervalos usados. El índice final de un intervalo queda excluido.

**Tu explicación:** …

## Actividad 3 · RGB y escala de grises (12 min)

Predice en qué canal destacará la marca roja. Usamos la misma escala de visualización (0–255) para comparar canales.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for k, nombre in enumerate(["Rojo", "Verde", "Azul"]):
    axes[k].imshow(imagen[:, :, k], cmap="gray", vmin=0, vmax=255)
    axes[k].set_title(nombre)
    axes[k].axis("off")
plt.show()
gris = np.asarray(Image.fromarray(imagen).convert("L"))
mostrar(gris, "Conversión a escala de grises", gris=True)
print("Forma en gris:", gris.shape)

**Modifica:** crea dos parches de distinto color y compara su aspecto en gris. La conversión mezcla canales: no es seleccionar el rojo. ¿Puede perderse información útil?

**Tu explicación:** …

## Actividad 4 · Recorte y resolución (15 min)

Un recorte conserva una región; reducir resolución resume información. Predice cuándo dejará de distinguirse la marca.

In [ ]:
recorte = imagen[80:140, 140:220]
mostrar(recorte, "Recorte alrededor de la marca")
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, tamano in zip(axes, [(320, 240), (40, 30), (10, 8)]):
    reducida = Image.fromarray(imagen).resize(tamano, Image.Resampling.BILINEAR)
    ax.imshow(reducida, interpolation="nearest")
    ax.set_title(f"{tamano[0]} × {tamano[1]}")
    ax.axis("off")
plt.show()

**Modifica:** añade una marca de 2 × 2 píxeles y repite. ¿Ampliar la imagen reducida recupera el detalle perdido? Justifica tu respuesta. Pillow recibe `(ancho, alto)`; NumPy informa `(alto, ancho, canales)`.

**Tu explicación:** …

## Actividad 5 · Valores y normalización (10 min)

Cambiar la escala numérica no aumenta la resolución. Primero convertimos a decimales para evitar operaciones accidentales con enteros de 8 bits.

In [ ]:
normalizada = imagen.astype(np.float32) / 255.0
print("Antes:", imagen.dtype, imagen[105, 180])
print("Después:", normalizada.dtype, normalizada[105, 180])
assert normalizada.min() >= 0 and normalizada.max() <= 1
mostrar(normalizada, "Valores entre 0 y 1")
# Ejemplo deliberado: uint8 vuelve a 0 al superar 255.
print("250 + 20 con uint8:", (np.array([250], dtype=np.uint8) + np.array([20], dtype=np.uint8))[0])
print("250 + 20 tras convertir:", (np.array([250], dtype=np.uint8).astype(np.float32) + 20)[0])

**Comprueba:** calcula el valor normalizado de 125. ¿La imagen contiene más detalle tras dividir por 255? Cada modelo exige su preparación específica; algunos usan otra normalización.

**Tu explicación:** …

## Actividad 6 · Ruido e iluminación (15 min)

El experimento simula cambios, no una cámara física. Mantén la referencia intacta y compara con ella.

In [ ]:
rng = np.random.default_rng(42)
factor_luz = 0.4
sigma = 25
oscura = np.clip(referencia.astype(np.float32) * factor_luz, 0, 255).astype(np.uint8)
ruido = rng.normal(0, sigma, referencia.shape)
ruidosa = np.clip(referencia.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, arr, titulo in zip(axes, [referencia, oscura, ruidosa], ["Referencia", "Menos luz", "Ruido"]):
    ax.imshow(arr)
    ax.set_title(titulo)
    ax.axis("off")
plt.show()

**Modifica:** prueba `sigma = 5, 25, 60` y factores de luz `0.2, 0.6, 1.4`. ¿En qué condiciones cuesta distinguir la marca? Si ejecutas de nuevo, la semilla permite repetir la misma realización de ruido.

**Tu explicación:** …

## Cierre y entrega (10 min)

Entrega una copia con tres experimentos: resolución, normalización y ruido/iluminación. Incluye una predicción, el resultado y una explicación de 2–3 frases en cada caso. Responde: ¿qué información pedirías de una cámara antes de diseñar una inspección de defectos pequeños?

**Autoevaluación:** puedo interpretar `shape`, localizar un píxel, distinguir RGB de gris, explicar qué pierde una reducción de resolución y preparar valores sin desbordamiento. Todavía no hemos entrenado un modelo de IA.